# Stable Diffusion test UI (SD 1.5 / SDXL)

1. **Runtime → Change runtime type → GPU (L4).**
2. Fill in the settings, then **Runtime → Run all**. The UI appears under the last cell and can be used while the model loads.

`MODEL_URL`: a Hugging Face file link (`https://huggingface.co/<org>/<repo>/blob/main/<file>.safetensors`) or a Civitai link with the version (`https://civitai.com/models/<id>?modelVersionId=<id>`). Pony/Illustrious checkpoints are `sdxl`.

Tokens are optional: add them to download gated/private models or to avoid anonymous downloads. Type them into the settings, or store them as **Colab Secrets** (🔑 sidebar) named `HF_TOKEN` / `CIVITAI_TOKEN` so they are never saved in the notebook. Don't share a copy with your tokens typed in.

In [ ]:
# @title 1 · Settings
MODEL_FAMILY = "sdxl"  # @param ["sd15", "sdxl"]
MODEL_URL = ""  # @param {type:"string"}
# @markdown Optional: your Hugging Face / Civitai API tokens (or Colab Secrets with these names).
HF_TOKEN = ""  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}
PORT = 8000  # @param {type:"integer"}

In [ ]:
# @title 2 · Install
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/sdmodeltester")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/ashwin2rai/sdmodeltester", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "notebooks"))
import colab_utils as cu  # noqa: E402

cu.install_requirements(REPO_DIR)
cu.run([sys.executable, "-m", "src.cli", "doctor", "--model-family", MODEL_FAMILY],
       "Environment check", cwd=REPO_DIR)
print("✓ Installed")

In [ ]:
# @title 3 · Download the model
MODEL_PATH = cu.fetch_checkpoint(
    MODEL_URL,
    REPO_DIR / "models",
    hf_token=cu.get_token("HF_TOKEN", HF_TOKEN),
    civitai_token=cu.get_token("CIVITAI_TOKEN", CIVITAI_TOKEN),
)
print(f"✓ {MODEL_PATH.name}")

In [ ]:
# @title 4 · Start the UI
cu.start_server([sys.executable, "-m", "src.cli", "serve", "--model", str(MODEL_PATH),
                 "--model-family", MODEL_FAMILY, "--port", str(PORT)], cwd=REPO_DIR, port=PORT)
cu.show_ui(PORT)